# Energy Reconstruction Using CNN - Both Charges

In [1]:
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt

from datetime import datetime

from tensorflow import keras
from keras import layers
from keras import models
from keras import callbacks

#from tensorflow.keras.models import Sequential
#from tensorflow.keras.layers import Dense, Conv2D, Flatten
#from tensorflow.keras.callbacks import ModelCheckpoint

from data_tools import load_preprocessed, dataPrep, nameModel

simPrefix = os.getcwd()+'\\simdata'

## Data Input

In [2]:
x, y = load_preprocessed(simPrefix, 'train')

Percentage of events with a NaN: 2.68


In [3]:
print(x.shape)
print(y.keys())
# each station has 2 tanks, each tank has 2 DOMs (high/log gain)
# each tank measures charge and time
# each station gives 2 charges and 2 times, 4 total pieces of data per station
# stations arranged in 10x10 square lattice, 2 corners of square unused
# charge measured in VEM, vertical equivalent muon

# 'dir' is true direction, rest of dir are reconstruted by simulations
# 'plane_dir' assumes shower is flat plane
# 'laputop_dir' performs likelihood analysis
# 'small_dir' compromises between plane and laputop

(549773, 10, 10, 4)
dict_keys(['comp', 'energy', 'dir', 'plane_dir', 'laputop_dir', 'small_dir'])


In [4]:
# 85/15 split for training/validation
energy = y['energy']
comp = y['comp']
theta, phi = y['dir'].transpose()
nevents = len(energy)
trainCut = (np.random.uniform(size=nevents) < 0.85)
testCut = np.logical_not(trainCut)

## Model Training

### Alpha Model
- Input: no charge merge, no time layers included, normalized data, combined with cosine of zenith angle
- Layers: Two convolutional layers for charge, then combined with zenith
- Output: Energy

In [5]:
# Name for model
key = 'q1q2Comp'
i = 0
while(os.path.exists('models/model_{}.h5'.format(key+str(i)))):
    i = i + 1
key = key+str(i)
numepochs = 10
# Data preparation: no merging of charge (q), no time layers included (t=False), data normalized from 0-1
prep = {'q':None, 't':False, 'normed':True}

In [6]:
key

'q1q2Comp4'

In [7]:
# Create model using functional API for multiple inputs
charge_input=keras.Input(shape=(10,10,2,), name="charge")

conv1_layer = layers.Conv2D(64, kernel_size=3, padding='same')(charge_input)
batch1_layer = layers.BatchNormalization()(conv1_layer) # default -> axis = -1, 
acti1_layer = layers.ReLU()(batch1_layer)
drop1_layer = layers.Dropout(0.2)(acti1_layer)

conv2_layer = layers.Conv2D(32, kernel_size=3, padding='same')(drop1_layer)
batch2_layer = layers.BatchNormalization()(conv2_layer) # default -> axis = -1, 
acti2_layer = layers.ReLU()(batch2_layer)
drop2_layer = layers.Dropout(0.2)(acti2_layer)

conv3_layer = layers.Conv2D(16, kernel_size=3, padding='same')(drop2_layer)
batch3_layer = layers.BatchNormalization()(conv3_layer) # default -> axis = -1, 
acti3_layer = layers.ReLU()(batch3_layer)
drop3_layer = layers.Dropout(0.2)(acti3_layer)

flat_layer=layers.Flatten()(drop3_layer)
dense1_layer = layers.Dense(128)(flat_layer)
batch4_layer = layers.BatchNormalization()(dense1_layer) # default -> axis = -1, 
acti4_layer = layers.ReLU()(batch4_layer)
drop4_layer = layers.Dropout(0.2)(acti4_layer)

dense2_layer = layers.Dense(128)(drop4_layer)
batch5_layer = layers.BatchNormalization()(dense2_layer) # default -> axis = -1, 
acti5_layer = layers.ReLU()(batch5_layer)
drop5_layer = layers.Dropout(0.2)(acti5_layer)

dense3_layer = layers.Dense(128)(drop5_layer)
batch6_layer = layers.BatchNormalization()(dense3_layer) # default -> axis = -1,
acti6_layer = layers.ReLU()(batch6_layer)
drop6_layer = layers.Dropout(0.2)(acti6_layer)

output = layers.Dense(1)(drop6_layer)

model = models.Model(inputs=[charge_input],outputs=output,name=key)

model.compile(loss='mean_squared_error', optimizer='adam', metrics=['mae','mse'])

## Old model used for reference
#model = Sequential(name=nameModel(prep, 'test'))  # Automatic naming for flexible assessment later
## Add model layers
#model.add(Conv2D(64, kernel_size=3, activation='relu', input_shape=(10,10,2)))
#model.add(Conv2D(32, kernel_size=3, activation='relu'))
#model.add(Flatten())
#model.add(Dense(1)) # No activation function for last layer of regression model

## Compile model
#model.compile(loss='mean_squared_error', optimizer='adam', metrics=['mae','mse'])

In [8]:
# Establish arrays to be trained on
x_i = dataPrep(x, y, **prep)
temp_y = energy

In [9]:
model.summary()

Model: "q1q2Comp4"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
charge (InputLayer)          [(None, 10, 10, 2)]       0         
_________________________________________________________________
conv2d (Conv2D)              (None, 10, 10, 64)        1216      
_________________________________________________________________
batch_normalization (BatchNo (None, 10, 10, 64)        256       
_________________________________________________________________
re_lu (ReLU)                 (None, 10, 10, 64)        0         
_________________________________________________________________
dropout (Dropout)            (None, 10, 10, 64)        0         
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 10, 10, 32)        18464     
_________________________________________________________________
batch_normalization_1 (Batch (None, 10, 10, 32)        12

In [10]:
keras.utils.plot_model(model,"model.png")

('You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) ', 'for plot_model/model_to_dot to work.')


In [11]:
# Train
csv_logger = callbacks.CSVLogger('models/{}'.format(key))
early_stop = callbacks.EarlyStopping(patience=5,restore_best_weights=True) # default -> val_loss4
checkpoint = callbacks.ModelCheckpoint('models/model_%s.h5' % key,save_best_only=True)
callbacklist = [early_stop, csv_logger]
history = model.fit(
    {"charge":x_i}, temp_y, epochs=numepochs,validation_split=0.15,callbacks=callbacklist)

Epoch 1/10
  922/14604 [>.............................] - ETA: 3:42 - loss: 3.2535 - mae: 1.1662 - mse: 3.2535

KeyboardInterrupt: 

In [ ]:
# Save model to file for easy loading
## WHERE ARE YOU SAVING TO?
np.save('models/model_%s.npy' % key,prep)
f = open("results.txt", "a")
now = datetime.now()
f.write("{}\t{}\tepochs:{}\tloss:{},{}\n".format(
    now.strftime("%m/%d/%Y %H:%M:%S"),
    key,
    len(history.history['loss']),
    np.min(history.history['loss']),
    np.min(history.history['val_loss'])
))
f.close()

In [ ]:
#model.save('models/model_%s.h5' % key)
prep = {'q':None, 't':False, 'normed':True, 'reco':'plane_'}
np.save('models/model_q1q2cosZComp0.npy',prep)